# Lab 19 — OpenAI Colab Runner

Notebook này là runner thực thi; notebook gốc vẫn được giữ nguyên làm reference của đề.

## Colab Secrets bắt buộc
- `OPENAI_API_KEY`
- `HF_TOKEN`
- `NEO4J_URI`
- `NEO4J_USER` = `neo4j`
- `NEO4J_PASSWORD`
- `NEO4J_DATABASE` = `neo4j`

Optional: `LLM_MODEL` và `JUDGE_MODEL` (mặc định `gpt-4.1-mini`). Nếu AuraDB chỉ dành riêng cho lab và muốn xóa graph cũ trước khi ingest, đặt `LAB_RESET_GRAPH=1`.

Flow: load first 5,000 rows → chạy definitions của notebook gốc → patch toàn bộ LLM calls sang OpenAI → `colab_solution.py` → official Golden 50.

In [ ]:
#@title 1 — Clone latest main (safe to rerun)
%cd /content
!rm -rf /content/lab19
!git clone -q https://github.com/QuocKhanhLuong/K3-Track3-Lab19-GraphRAG-2A202601713-LuongQuocKhanh.git /content/lab19
%cd /content/lab19
!git log -1 --oneline


In [ ]:
#@title 2 — Execute reference notebook definitions on the official first-5000 scope
import json
from pathlib import Path

reference_path = Path('/content/lab19/Day19_GraphRAG_vs_FlatRAG_Production_Lab_Guide.ipynb')
nb = json.loads(reference_path.read_text(encoding='utf-8'))

for idx, cell in enumerate(nb['cells']):
    if cell.get('cell_type') != 'code':
        continue
    src = ''.join(cell.get('source', []))
    src = src.replace('LIMIT_ROWS = 1_000_000', 'LIMIT_ROWS = 5000')
    src = src.replace('LIMIT_MB = 300', 'LIMIT_MB = 80')
    src = src.replace('PRIORITIZE_MB = True', 'PRIORITIZE_MB = False')
    result = get_ipython().run_cell(src)
    if getattr(result, 'error_before_exec', None):
        raise result.error_before_exec
    if getattr(result, 'error_in_exec', None):
        raise result.error_in_exec

print('Reference definitions loaded. Dataset file:', DATA_PATH)


In [ ]:
#@title 3 — OpenAI patch + full GraphRAG/FlatRAG solution
%run -i /content/lab19/openai_runtime_patch.py
test_text, test_usage = groq_chat(
    [{'role': 'user', 'content': 'Reply exactly: OK'}],
    model=GROQ_MODEL,
)
print('OpenAI smoke test:', test_text, test_usage)
assert test_text.strip().upper().startswith('OK')
%run -i /content/lab19/colab_solution.py


In [ ]:
#@title 4 — Official Golden Dataset: 50 questions / first 5,000 rows
%run -i /content/lab19/official_golden_eval.py


## Sau khi chạy xong
Cell 3 tạo baseline artifacts; Cell 4 ghi đè hai CSV rubric chính bằng kết quả official 50-question benchmark và tải `/content/lab19_submission_official50.zip`. Kiểm tra `outputs/extraction_errors.csv` và `reports/lab_report.md` trước khi commit output về repo.